In [13]:
def decomp(x,word_size,already_decomp):
    
    if x in already_decomp:
        return already_decomp[x]
    
    X = Integer(x).digits(2)
    X += [0]*(word_size-len(X))
    
    X_e = [X[even] for even in range(0,word_size,2)]
    X_o = [X[odd] for odd in range(1,word_size,2)]
    
    x_e = sum([4^i*X_e[i] for i in range(word_size//2)])
    x_o = sum([4^i*X_o[i] for i in range(word_size//2)])
    
    already_decomp[x] = (x_e,x_o)
    return x_e,x_o

In [2]:
def value(Trace_BitW,a,b=None):
    if b == None :
        return Trace_BitW[a]
    if type(b)==Integer:
        return Trace_BitW[a],b
    return Trace_BitW[a],Trace_BitW[b]

In [3]:
def get_inputs_nodes(node,var_calc):

    a = node.incoming[0]
    b = _
    if len(node.incoming)>1 :
        b = node.incoming[1]
        
    if a in var_calc:
        a = var_calc[a]
        
    if b in var_calc:
        b = var_calc[b]

    return a,b

In [ ]:
def decomp_sum(a,b,word_size,dict_decomp):
    
    a_e,a_o = decomp(a,word_size,dict_decomp)
    b_e,b_o = decomp(b,word_size,dict_decomp)
    e_e,e_o = decomp(a_e+b_e,word_size,dict_decomp)
    o_e,o_o = decomp(a_o+b_o,word_size,dict_decomp)
    
    return e_e,e_o,o_e,o_o

In [15]:
def find_witness_AND_XOR_OR(node,a,b,word_size,Trace_BitW,dict_decomp,already_compute,var_calc):
    print("AND/XOR/OR")
    if (a,b) in already_compute :
        var_calc[node] = already_compute[(a,b)]
        print("The AND/XOR/OR gate is already computed, no proof to provide")
        return ()
    already_compute[(a,b)] = node
    
    a,b = value(Trace_BitW,a,b)
         
    return decomp_sum(a,b,word_size,dict_decomp)

In [7]:
def find_witness_NOT(node,a,word_size,Trace_BitW,dict_decomp,already_compute_NOT,already_compute_AND_XOR_OR,var_calc):
    print("NOT")
    if a in already_compute_NOT :
        var_calc[node] = already_compute_NOT[(a,b)]
        print("The NOT gate is already computed, no proof to provide")
        return ()
    already_compute_NOT[a] = node
    
    return find_witness_AND_XOR_OR(node,a,2^word_size-1,word_size,Trace_BitW,dict_decomp,already_compute_AND_XOR_OR,var_calc)

In [ ]:
def bin_c(c,word_size):
    if(c>=2^word_size):
        k = 1
    else :
        k = 0
    return k

In [ ]:
def find_witness_ADD(node,a,b,word_size,Trace_BitW,dict_decomp,already_compute,var_calc):
    print("ADD")
    if (a,b) in already_compute:
        var_calc[node] = already_compute[(a,b)]
        print("The ADD gate is already computed, no proof to provide")
        return ()
    already_compute[(a,b)] = node
    
    a,b = value(Trace_BitW,a,b)
    
    c = a+b
    c_e,c_o = decomp(c,word_size,dict_decomp)
    
    return c_e,c_o,bin_c(a+b,word_size)

In [ ]:
def find_witness_SUB(node,a,b,word_size,Trace_BitW,dict_decomp,already_compute,var_calc):
    print("SUB")
    if (a,b) in already_compute:
        var_calc[node] = already_compute[(a,b)]
        print("The SUB gate is already computed, no proof to provide")
        return ()
    already_compute[(a,b)] = node
    
    a,b = value(Trace_BitW,a,b)
    
    c = (a-b) %2^word_size
    
    c_e,c_o = decomp(c,word_size,dict_decomp)
    
    return c_e,c_o,bin_c(b+c,word_size)

In [ ]:
def LK_c(c,word_size,dict_decomp):
    c_e,c_o = decomp(c,word_size,dict_decomp)

    k = c //2^word_size

    k_e,k_o = decomp(k,word_size,dict_decomp)

    return c_e,c_o,k_e,k_o

In [ ]:
def find_witness_MUL(node,a,b,word_size,Trace_BitW,dict_decomp,already_compute,var_calc):
    print("MUL")
    if (a,b) in already_compute:
        var_calc[node] = already_compute[(a,b)]
        print("The MUL gate is already computed, no proof to provide")
        return ()
    already_compute[(a,b)] = node
    
    a,b = value(Trace_BitW,a,b)
    
    return LK_c(a*b,word_size,dict_decomp)

In [ ]:
def div_mod_wit(a,b,word_size,dict_decomp):
    q,r = divmod(a,b)

    q_e,q_o = decomp(q,word_size,dict_decomp)
    r_e,r_o = decomp(r,word_size,dict_decomp)

    b_min_r_min_1 = b-r-1
    b_min_r_min_1_e,b_min_r_min_1_o = decomp(b_min_r_min_1,word_size,dict_decomp)

    return q_e,q_o,r_e,r_o,b_min_r_min_1_e,b_min_r_min_1_o

In [ ]:
def find_witness_DIV_MOD(node,a,b,word_size,Trace_BitW,dict_decomp,already_compute,var_calc):
    if (a,b) in already_compute:
        var_calc[node] = already_compute[(a,b)]
        return ()
    already_compute[(a,b)] = node
    
    a,b = value(Trace_BitW,a,b)

    return div_mod_wit(a,b,word_size,dict_decomp)

In [16]:
def find_witness_SHL(node,a,b,word_size,Trace_BitW,dict_decomp,already_compute_SHL,already_compute_MUL,var_calc):
    if b>=word_size :
        return ()
    
    if (a,b) in already_compute_SHL:
        var_calc[node] = already_compute_SHL[(a,b)]
        return ()
    already_compute_SHL[(a,b)] = node
    
    return find_witness_MUL(node,a,2^b,word_size,Trace_BitW,dict_decomp,already_compute_MUL,var_calc)

In [ ]:
def find_witness_SHR(node,a,b,word_size,Trace_BitW,dict_decomp,already_compute,var_calc):
    if b>=word_size :
        return ()
    
    if (a,b) in already_compute:
        var_calc[node] = already_compute[(a,b)]
        return ()
    
    already_compute[(a,b)] = node
    
    a = value(Trace_BitW,a)
    
    return div_mod_wit(a,2^b,word_size,dict_decomp)